# Build Hybrid RAG Index on Kaggle

Этот ноутбук собирает retrieval-индекс из полного raw corpus: Cloudflare crawl + manual docs, прогоняет eval и упаковывает артефакты для возврата в локальный проект.

Ожидается, что в Kaggle загружен весь каталог `data/raw`.


## Что нужно добавить в Kaggle заранее

1. Dataset с полным raw corpus: локальный каталог `data/raw/`.
2. Код проекта: либо `git clone` из GitHub при включенном интернете, либо отдельный dataset/zip с репозиторием.
3. Если нужен OCR, включить internet и установить `tesseract-ocr` через `apt`.


In [ ]:
from pathlib import Path

RAW_DATASET_DIR = Path('/kaggle/input/guap-raw')
RAW_RECORDS = RAW_DATASET_DIR / 'cloudflare' / 'latest' / 'records.jsonl'
MANUAL_DOCS_DIR = RAW_DATASET_DIR / 'manual_docs'
FIRECRAWL_DIR = RAW_DATASET_DIR / 'firecrawl'
REPO_DIR = Path('/kaggle/working/llm-speaker-core')

print('raw records:', RAW_RECORDS)
print('manual docs dir:', MANUAL_DOCS_DIR)
print('firecrawl dir:', FIRECRAWL_DIR)

assert RAW_RECORDS.exists(), f'Missing raw records: {RAW_RECORDS}'


In [ ]:
# Если интернет включен, клонируем репозиторий.
if not REPO_DIR.exists():
    !git clone https://github.com/chudinovAI/llm-speaker-core.git {REPO_DIR}
%cd {REPO_DIR}


In [ ]:
# Минимальные retrieval-зависимости. Voice stack здесь не нужен.
!python -m pip install -q --upgrade pip
!python -m pip install -q faiss-cpu huggingface-hub pymupdf pytesseract pypdf python-docx sentence-transformers FlagEmbedding requests numpy torch transformers pytest mypy


In [ ]:
# Опционально: OCR fallback
# !apt-get update -qq && apt-get install -y -qq tesseract-ocr


In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))

from llm_speaker_core.retrieval.build import build_hybrid_index

report = build_hybrid_index(
    raw_records=RAW_RECORDS,
    documents_out=REPO_DIR / 'data/normalized/documents.jsonl',
    chunks_out=REPO_DIR / 'data/normalized/chunks.jsonl',
    manifest_out=REPO_DIR / 'data/index_manifest.json',
    lexical_out=REPO_DIR / 'data/indexes/bm25/index.json',
    dense_out=REPO_DIR / 'data/indexes/faiss/index.json',
    embedding_model='BAAI/bge-m3',
    reranker_model='BAAI/bge-reranker-v2-m3',
    manual_docs_dir=MANUAL_DOCS_DIR,
    firecrawl_dir=FIRECRAWL_DIR,
)
report


In [ ]:
from llm_speaker_core.retrieval.eval import evaluate_manifest

gold_path = REPO_DIR / 'data/eval/gold_queries.jsonl'
eval_report = evaluate_manifest(
    manifest_path=REPO_DIR / 'data/index_manifest.json',
    gold_path=gold_path,
    output_path=REPO_DIR / 'data/eval/hybrid_eval_report.json',
)
eval_report


In [ ]:
from llm_speaker_core.retrieval.service import HybridRetrievalService

service = HybridRetrievalService.load(REPO_DIR / 'data/index_manifest.json')
for query in [
    'Как поступить в ГУАП?',
    'Сколько стоит обучение в ГУАП?',
    'Как связаться с приемной комиссией?',
    'Какие есть студенческие активности?',
]:
    hits = service.search_hits(query, top_k=3)
    print('\nQUERY:', query)
    for hit in hits:
        print('-', hit.source, hit.score, hit.retrieval_stage)


In [ ]:
%cd {REPO_DIR}
!zip -r /kaggle/working/rag_artifacts_kaggle.zip data/index_manifest.json data/indexes data/normalized data/eval/hybrid_eval_report.json
print('/kaggle/working/rag_artifacts_kaggle.zip')
